# Kontinuitet i nestacionarni spremnik — predvidi, izračunaj, provjeri

**Poglavlje U07: kinematika, kontinuitet i Reynoldsov transportni teorem**

Najprije zadržavamo osnovni scenarij suženja cijevi. Zatim isti zakon
očuvanja mase primjenjujemo na nestacionarni kontrolni volumen spremnika i
provjeravamo osjetljivost eksplicitne metode na vremenski korak.


## 1. Predvidi

1. Ako se promjer prepolovi uz isti $Q$, koliko puta raste srednja brzina?
2. U spremniku je $Q_{in}$ stalan, a $Q_{out}=k\sqrt h$. Hoće li se razina
   iz početnih 0,25 m približavati ili udaljavati od ravnoteže 0,36 m?
3. Hoće li prevelik vremenski korak promijeniti bilancu mase ili samo točnost
   aproksimacije kontinuiranog rješenja?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def suzenje(D1_m, D2_m, Q_m3_s):
    A1, A2 = np.pi*D1_m**2/4, np.pi*D2_m**2/4
    return {"A1": A1, "A2": A2, "v1": Q_m3_s/A1, "v2": Q_m3_s/A2}

Q, D1, D2 = 0.012, 0.120, 0.060
st = suzenje(D1, D2, Q)
print(f"v1 = {st['v1']:.3f} m/s, v2 = {st['v2']:.3f} m/s")
print(f"v2/v1 = {st['v2']/st['v1']:.3f}")


## 2. Izračunaj — nestacionarni konačni volumen spremnika

Za spremnik stalne tlocrtne površine $A_T$ bilanca volumena glasi

$$A_T\frac{dh}{dt}=Q_{in}-k\sqrt h.$$

Eksplicitni Eulerov korak je
$h^{n+1}=h^n+\Delta t(Q_{in}-k\sqrt{h^n})/A_T$.
Kao precizno referentno rješenje koristimo RK4 s vrlo malim korakom.


In [ ]:
A_T, Q_in, k_izlaz, h0, T = 6.0, 0.006, 0.010, 0.25, 1800.0

def rhs(h):
    return (Q_in-k_izlaz*np.sqrt(max(h, 0.0)))/A_T

def euler(dt):
    n = int(round(T/dt))
    dt = T/n
    t = np.linspace(0.0, T, n+1)
    h = np.empty(n+1)
    h[0] = h0
    q_out = np.empty(n)
    for j in range(n):
        q_out[j] = k_izlaz*np.sqrt(max(h[j], 0.0))
        h[j+1] = h[j] + dt*(Q_in-q_out[j])/A_T
    maseni_debalans = A_T*(h[-1]-h0) - np.sum((Q_in-q_out)*dt)
    return t, h, maseni_debalans

def rk4(dt):
    n = int(round(T/dt))
    dt = T/n
    h = h0
    for _ in range(n):
        k1 = rhs(h)
        k2 = rhs(h+0.5*dt*k1)
        k3 = rhs(h+0.5*dt*k2)
        k4 = rhs(h+dt*k3)
        h += dt*(k1+2*k2+2*k3+k4)/6
    return h

h_ref = rk4(0.05)
dt_mreza = np.array([20.0, 10.0, 5.0, 2.5, 1.25])
izvodi = [euler(dt) for dt in dt_mreza]
h_kraj = np.array([rez[1][-1] for rez in izvodi])
err = np.abs(h_kraj-h_ref)
red = np.log2(err[:-1]/err[1:])
print(f"RK4 referenca h(T) = {h_ref:.9f} m")
print("dt [s]   h_Euler(T) [m]   pogreška [m]   opaženi red")
for i, (dt, hk, e) in enumerate(zip(dt_mreza, h_kraj, err)):
    p = "--" if i == 0 else f"{red[i-1]:.3f}"
    print(f"{dt:6.2f} {hk:16.9f} {e:14.3e} {p:>13s}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 3.8))
for dt, (t, h, _) in zip(dt_mreza[:3], izvodi[:3]):
    ax1.plot(t/60, h, label=fr"$\Delta t={dt:g}$ s")
ax1.axhline((Q_in/k_izlaz)**2, color="#2e7d32", ls="--", label="ravnoteža")
ax1.set(xlabel="vrijeme (min)", ylabel="razina h (m)")
ax1.grid(ls=":", alpha=0.5)
ax1.legend()
ax2.loglog(dt_mreza, err, "o-")
ax2.set(xlabel=r"vremenski korak $\Delta t$ (s)", ylabel="pogreška h(T) (m)",
        title="Provjera vremenskog koraka")
ax2.grid(ls=":", which="both", alpha=0.5)
fig.tight_layout()
plt.show()


## 3. Provjeri — dvije bilance i konvergencija

U suženju posebno uspoređujemo ulazni i izlazni protok. U spremniku
integrirani dotok minus istok mora biti jednak akumulaciji u svakom
diskretnom računu. To ne znači da je vremenska diskretizacija bez pogreške:
usporedba s RK4 pokazuje konvergenciju prema kontinuiranom rješenju.


In [ ]:
Q1 = st["A1"]*st["v1"]
Q2 = st["A2"]*st["v2"]
debalansi = np.array([rez[2] for rez in izvodi])
h_eq = (Q_in/k_izlaz)**2

assert np.isclose(Q1, Q2, rtol=1e-13)
assert np.max(np.abs(debalansi)) < 1e-12
assert np.all((red[-3:] > 0.98) & (red[-3:] < 1.02))
assert h0 < h_ref < h_eq
print("PASS: kontinuitet suženja, FV bilanca, prvi red i prilaz ravnoteži.")


## Granica modela

Model koristi jednoliku razinu, nestlačivu tekućinu i kvazistacionarni zakon
istjecanja s konstantnim koeficijentom $k$. Ne opisuje valove u spremniku,
promjenjivu geometriju, inerciju u cijevi ni kavitaciju. Za pokretni kontrolni
volumen protok bi trebalo zapisati relativnom brzinom kroz granicu.
